In [5]:
import pandas as pd

df = pd.read_csv("../data/raw/IPL.csv")

print(df.shape)
df.head()

C:\Users\Shiva\AppData\Local\Temp\ipykernel_2712\2085303613.py:3: DtypeWarning: Columns (0: review_batter, 1: team_reviewed, 2: review_decision, 3: umpire, 4: season, 5: superover_winner, 6: result_type, 7: method, 8: event_match_no) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/raw/IPL.csv")


(278205, 64)


,Unnamed: 0,match_id,date,match_type,event_name,innings,batting_team,bowling_team,over,ball,...,team_runs,team_balls,team_wicket,new_batter,batter_runs,batter_balls,bowler_wicket,batting_partners,next_batter,striker_out
0,131970,335982,2008-04-18,T20,Indian Premier League,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,1,...,1,1,0,NaN,0,1,0,"('BB McCullum', 'SC Ganguly')",NaN,False
1,131971,335982,2008-04-18,T20,Indian Premier League,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,2,...,1,2,0,NaN,0,1,0,"('BB McCullum', 'SC Ganguly')",NaN,False
2,131972,335982,2008-04-18,T20,Indian Premier League,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,...,2,2,0,NaN,0,1,0,"('BB McCullum', 'SC Ganguly')",NaN,False
3,131973,335982,2008-04-18,T20,Indian Premier League,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,...,2,3,0,NaN,0,2,0,"('BB McCullum', 'SC Ganguly')",NaN,False
4,131974,335982,2008-04-18,T20,Indian Premier League,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,4,...,2,4,0,NaN,0,3,0,"('BB McCullum', 'SC Ganguly')",NaN,False


In [2]:
df = df[[
    'match_id',
    'date',
    'batting_team',
    'bowling_team',
    'venue',
    'batter',
    'bowler',
    'runs_total',
    'wicket_kind'
]]

In [3]:
df.head()

,match_id,date,batting_team,bowling_team,venue,batter,bowler,runs_total,wicket_kind
0,335982,2008-04-18,Kolkata Knight Riders,Royal Challengers Bangalore,M Chinnaswamy Stadium,SC Ganguly,P Kumar,1,NaN
1,335982,2008-04-18,Kolkata Knight Riders,Royal Challengers Bangalore,M Chinnaswamy Stadium,BB McCullum,P Kumar,0,NaN
2,335982,2008-04-18,Kolkata Knight Riders,Royal Challengers Bangalore,M Chinnaswamy Stadium,BB McCullum,P Kumar,1,NaN
3,335982,2008-04-18,Kolkata Knight Riders,Royal Challengers Bangalore,M Chinnaswamy Stadium,BB McCullum,P Kumar,0,NaN
4,335982,2008-04-18,Kolkata Knight Riders,Royal Challengers Bangalore,M Chinnaswamy Stadium,BB McCullum,P Kumar,0,NaN


In [4]:
df.isnull().sum()

match_id             0
date                 0
batting_team         0
bowling_team         0
venue                0
batter               0
bowler               0
runs_total           0
wicket_kind     264382
dtype: int64

In [5]:
df['wicket_kind'] = df['wicket_kind'].fillna('None')

In [6]:
df['date'] = pd.to_datetime(df['date'])

In [6]:
df['batting_team'] = df['batting_team'].replace({
    'Delhi Daredevils': 'Delhi Capitals',
    'Kings XI Punjab': 'Punjab Kings'
})

df['bowling_team'] = df['bowling_team'].replace({
    'Delhi Daredevils': 'Delhi Capitals',
    'Kings XI Punjab': 'Punjab Kings'
})

In [7]:
team_scores = df.groupby(['match_id', 'batting_team'])['runs_total'].sum().reset_index()

In [8]:
team_scores.head()

,match_id,batting_team,runs_total
0,335982,Kolkata Knight Riders,222
1,335982,Royal Challengers Bangalore,82
2,335983,Chennai Super Kings,240
3,335983,Punjab Kings,207
4,335984,Delhi Capitals,132


In [9]:
team_scores.to_csv("../data/processed/team_scores.csv", index=False)

In [10]:
match_df = team_scores.pivot(index='match_id', columns='batting_team', values='runs_total').reset_index()

In [11]:
match_df.head()

batting_team,match_id,Chennai Super Kings,Deccan Chargers,Delhi Capitals,Gujarat Lions,Gujarat Titans,Kochi Tuskers Kerala,Kolkata Knight Riders,Lucknow Super Giants,Mumbai Indians,Pune Warriors,Punjab Kings,Rajasthan Royals,Rising Pune Supergiant,Rising Pune Supergiants,Royal Challengers Bangalore,Royal Challengers Bengaluru,Sunrisers Hyderabad
0,335982,NaN,NaN,NaN,NaN,NaN,NaN,222.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,82.0,NaN,NaN
1,335983,240.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,207.0,NaN,NaN,NaN,NaN,NaN,NaN
2,335984,NaN,NaN,132.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,129.0,NaN,NaN,NaN,NaN,NaN
3,335985,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,165.0,NaN,NaN,NaN,NaN,NaN,166.0,NaN,NaN
4,335986,NaN,110.0,NaN,NaN,NaN,NaN,112.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
def process_match(x):
    # Only keep matches with exactly 2 teams
    if len(x) != 2:
        return None
    
    team1 = x.iloc[0]['batting_team']
    team2 = x.iloc[1]['batting_team']
    
    score1 = x.iloc[0]['runs_total']
    score2 = x.iloc[1]['runs_total']
    
    winner = team1 if score1 > score2 else team2
    
    return pd.Series({
        'team1': team1,
        'team2': team2,
        'score1': score1,
        'score2': score2,
        'winner': winner
    })

match_df = team_scores.groupby('match_id').apply(process_match).dropna().reset_index()

In [18]:
match_df.head()

,match_id,team1,team2,score1,score2,winner
0,335982,Kolkata Knight Riders,Royal Challengers Bangalore,222.0,82.0,Kolkata Knight Riders
1,335983,Chennai Super Kings,Punjab Kings,240.0,207.0,Chennai Super Kings
2,335984,Delhi Capitals,Rajasthan Royals,132.0,129.0,Delhi Capitals
3,335985,Mumbai Indians,Royal Challengers Bangalore,165.0,166.0,Royal Challengers Bangalore
4,335986,Deccan Chargers,Kolkata Knight Riders,110.0,112.0,Kolkata Knight Riders


In [19]:
match_df.to_csv("../data/processed/match_results.csv", index=False)

In [ ]:
#Creating players Runs per match

In [20]:
player_runs = df.groupby(['match_id', 'batter'])['runs_total'].sum().reset_index()

In [24]:
player_runs.head()

,match_id,batter,runs_total
0,335982,AA Noffke,11
1,335982,B Akhil,0
2,335982,BB McCullum,169
3,335982,CL White,6
4,335982,DJ Hussey,12


In [25]:
player_runs = player_runs.sort_values(by=['batter', 'match_id'])

In [26]:
player_runs['form'] = player_runs.groupby('batter')['runs_total'].rolling(5).mean().reset_index(0, drop=True)

In [29]:

player_runs.head(10)

,match_id,batter,runs_total,form
4299,548346,A Ashish Reddy,10,NaN
4390,548352,A Ashish Reddy,3,NaN
4496,548359,A Ashish Reddy,12,NaN
4699,548373,A Ashish Reddy,10,NaN
4747,548376,A Ashish Reddy,5,8.0
4866,598000,A Ashish Reddy,7,7.4
4933,598004,A Ashish Reddy,14,9.6
5027,598010,A Ashish Reddy,16,10.4
5076,598013,A Ashish Reddy,4,9.2
5160,598018,A Ashish Reddy,19,12.0


In [30]:
player_runs['form'] = player_runs['form'].fillna(0)

In [31]:
player_runs.to_csv("../data/processed/player_form.csv", index=False)